# 03 - Avaliacao e visualizacao

Neste notebook, utilizamos a melhor configuração identificada no Notebook 2 para gerar a solução final da amostra de entregas. O objetivo e avaliar o comportamento das rotas obtidas e produzir os artefatos visuais que serao aproveitados no relatorio técnico e no vídeo.

## 1. Carregamento da amostra oficial e da frota de experimento

Nesta etapa, carregamos a amostra final de entregas e a frota recalibrada definida no Notebook 2. Esses arquivos passam a compor a base da avaliação final das rotas.

In [1]:
from pathlib import Path
import pandas as pd
import sys

BASE_DIR = Path("..").resolve()
SAMPLES_DIR = BASE_DIR / "dataset" / "samples"
SRC_DIR = BASE_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from médical_route_optimizer import load_stops, load_vehicles, optimize_routes
from médical_route_optimizer.visualization import plot_history, plot_routes

stops_path = SAMPLES_DIR / "stops_sample_100.csv"
vehicles_path = SAMPLES_DIR / "vehicles_sample_100_experiment.csv"

stops_df = pd.read_csv(stops_path)
vehicles_df = pd.read_csv(vehicles_path)

stops = load_stops(stops_path)
vehicles = load_vehicles(vehicles_path)

print("Stops shape:", stops_df.shape)
print("Vehicles shape:", vehicles_df.shape)

display(stops_df.head())
display(vehicles_df)

Stops shape: (101, 7)
Vehicles shape: (4, 3)


,stop_id,name,x,y,demand,priority,is_depot
0,DEPOT-STORE,Store Depot,11.003669,76.976494,0,0,True
1,cobx423154877,médicamentos_críticos,17.501028,78.419645,2,5,False
2,gbbd486976829,médicamentos_críticos,22.340526,73.200937,2,5,False
3,mpmj126993214,médicamentos_críticos,13.023041,77.793237,2,5,False
4,opjk190252636,médicamentos_críticos,19.014049,72.845203,2,5,False


,vehicle_id,capacity,max_distance
0,motorcycle,65,140
1,van,85,210
2,scooter,65,140
3,bicycle,45,120


## 2. Configuracao final adotada

A configuração `exp_3_more_population`, escolhida no Notebook 2, será usada como base da avaliação final. Essa configuração apresentou o menor fitness entre os experimentos comparados.

In [2]:
final_config = {
    "population_size": 120,
    "generations": 120,
    "mutation_rate": 0.10,
    "elite_size": 2,
    "random_seed": 42,
}

final_config

{'population_size': 120,
 'generations': 120,
 'mutation_rate': 0.1,
 'elite_size': 2,
 'random_seed': 42}

## 3. Execucao da configuração final

Nesta etapa, executamos a configuração final selecionada para produzir a solução que será analisada neste notebook. O foco agora passa a ser a leitura operacional das rotas e a produção dos artefatos de apoio.

In [7]:
final_result = optimize_routes(
    stops=stops,
    vehicles=vehicles,
    population_size=final_config["population_size"],
    generations=final_config["generations"],
    mutation_rate=final_config["mutation_rate"],
    elite_size=final_config["elite_size"],
    random_seed=final_config["random_seed"],
)

print("Best fitness:", final_result.best_fitness)
print("Total distance:", final_result.total_distance)
print("Historico de geracoes:", len(final_result.history))

Best fitness: 31519.0
Total distance: 605.7
Historico de geracoes: 120


## 4. Resumo das rotas por veículo

Após a execucao da configuração final, consolidamos as principais informações de cada rota para avaliar distribuição de carga, distância percorrida e possiveis excessos residuais.

In [8]:
final_route_summary = pd.DataFrame(
    [
        {
            "vehicle_id": route.vehicle_id,
            "num_stops": len(route.stop_ids),
            "distance": route.distance,
            "load": route.load,
            "capacity_overflow": route.capacity_overflow,
            "distance_overflow": route.distance_overflow,
        }
        for route in final_result.routes
    ]
)

display(final_route_summary)

,vehicle_id,num_stops,distance,load,capacity_overflow,distance_overflow
0,motorcycle,25,135.43,64,0,0.00
1,van,30,209.77,82,0,0.00
2,scooter,25,140.83,65,0,0.83
3,bicycle,20,119.67,46,0,0.00


## 5. Ajuste residual da frota final

A avaliação inicial da solução final mostrou um excesso residual de carga no veículo `bicycle`. Para consolidar uma solução operacionalmente viavel, foi realizado um ajuste pontual na capacidade desse veículo, sem alterar a estrutura geral da frota.

In [11]:
vehicles_df.loc[vehicles_df["vehicle_id"] == "bicycle", "capacity"] = 46

display(vehicles_df)

,vehicle_id,capacity,max_distance
0,motorcycle,65,140
1,van,85,210
2,scooter,65,140
3,bicycle,46,120


## 6. Reexecucao da solução final com frota ajustada

Após o ajuste residual da frota, reexecutamos a configuração final para garantir que as tabelas e figuras exportadas representem a versao definitiva da solução analisada neste notebook.

In [12]:
vehicles_path = SAMPLES_DIR / "vehicles_sample_100_experiment.csv"
vehicles_df.to_csv(vehicles_path, index=False)

vehicles = load_vehicles(vehicles_path)

final_result = optimize_routes(
    stops=stops,
    vehicles=vehicles,
    population_size=final_config["population_size"],
    generations=final_config["generations"],
    mutation_rate=final_config["mutation_rate"],
    elite_size=final_config["elite_size"],
    random_seed=final_config["random_seed"],
)

final_route_summary = pd.DataFrame(
    [
        {
            "vehicle_id": route.vehicle_id,
            "num_stops": len(route.stop_ids),
            "distance": route.distance,
            "load": route.load,
            "capacity_overflow": route.capacity_overflow,
            "distance_overflow": route.distance_overflow,
        }
        for route in final_result.routes
    ]
)

print("Best fitness:", final_result.best_fitness)
print("Total distance:", final_result.total_distance)
display(final_route_summary)


Best fitness: 31519.0
Total distance: 605.7


,vehicle_id,num_stops,distance,load,capacity_overflow,distance_overflow
0,motorcycle,25,135.43,64,0,0.00
1,van,30,209.77,82,0,0.00
2,scooter,25,140.83,65,0,0.83
3,bicycle,20,119.67,46,0,0.00


## 7. Comparacao visual dos experimentos

Antes de consolidar a solução final, vale registrar visualmente o comportamento dos três experimentos comparados no Notebook 2. O objetivo aqui é mostrar como fitness e distância total variaram entre as configurações testadas.


In [ ]:
import matplotlib.pyplot as plt

FIGURES_DIR = BASE_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

experiment_comparison_df = pd.DataFrame(
    [
        {"experiment": "exp_1_base", "best_fitness": 31519.40, "total_distance": 575.40},
        {"experiment": "exp_2_more_generations", "best_fitness": 31419.74, "total_distance": 580.94},
        {"experiment": "exp_3_more_population", "best_fitness": 31403.70, "total_distance": 583.70},
    ]
)

display(experiment_comparison_df.sort_values("best_fitness"))

comparison_figure_path = FIGURES_DIR / "comparacao-experimentos-final.png"
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(experiment_comparison_df["experiment"], experiment_comparison_df["best_fitness"], color="#4C78A8")
axes[0].set_title("Fitness por experimento")
axes[0].set_ylabel("Best fitness")
axes[0].tick_params(axis="x", rotation=20)

axes[1].bar(experiment_comparison_df["experiment"], experiment_comparison_df["total_distance"], color="#F58518")
axes[1].set_title("Distância total por experimento")
axes[1].set_ylabel("Total distance")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.savefig(comparison_figure_path, dpi=150)
plt.show()

print("Figura de comparação salva em:", comparison_figure_path)


## 8. Distancia e carga por veiculo

Nesta etapa, transformamos a tabela-resumo das rotas em gráficos simples de barras. Esses visuais ajudam a mostrar, de forma direta, como a solução distribuiu esforço operacional e carga entre os veículos da frota.


In [ ]:
distance_figure_path = FIGURES_DIR / "distancia-por-veiculo-final.png"
load_figure_path = FIGURES_DIR / "carga-por-veiculo-final.png"

distance_plot = final_route_summary.sort_values("distance", ascending=False)
load_plot = final_route_summary.sort_values("load", ascending=False)

plt.figure(figsize=(8, 4))
plt.bar(distance_plot["vehicle_id"], distance_plot["distance"], color="#4C78A8")
plt.title("Distância por veículo")
plt.ylabel("Distância total")
plt.tight_layout()
plt.savefig(distance_figure_path, dpi=150)
plt.show()

plt.figure(figsize=(8, 4))
plt.bar(load_plot["vehicle_id"], load_plot["load"], color="#54A24B")
plt.title("Carga por veículo")
plt.ylabel("Carga total")
plt.tight_layout()
plt.savefig(load_figure_path, dpi=150)
plt.show()

print("Figura de distância salva em:", distance_figure_path)
print("Figura de carga salva em:", load_figure_path)


## 9. Distribuicao da amostra final

Além do desempenho das rotas, também é útil visualizar o perfil da amostra final utilizada no experimento. Os gráficos abaixo resumem a composição por classe de entrega e por prioridade.


In [ ]:
sample_deliveries_df = stops_df[~stops_df["is_depot"]].copy()

delivery_distribution = sample_deliveries_df.groupby("name").size().reset_index(name="count")
priority_distribution = sample_deliveries_df.groupby("priority").size().reset_index(name="count")

distribution_figure_path = FIGURES_DIR / "distribuicao-amostra-final.png"
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(delivery_distribution["name"], delivery_distribution["count"], color="#E45756")
axes[0].set_title("Entregas por classe")
axes[0].set_ylabel("Quantidade")
axes[0].tick_params(axis="x", rotation=15)

axes[1].bar(priority_distribution["priority"].astype(str), priority_distribution["count"], color="#72B7B2")
axes[1].set_title("Entregas por prioridade")
axes[1].set_ylabel("Quantidade")

plt.tight_layout()
plt.savefig(distribution_figure_path, dpi=150)
plt.show()

display(delivery_distribution)
display(priority_distribution)
print("Figura de distribuição salva em:", distribution_figure_path)


## 10. Exportacao das figuras finais

Com a solução final consolidada, exportamos os gráficos principais de convergência do algoritmo genético e de visualização das rotas. Esses artefatos serão utilizados na etapa de documentação e apresentação do projeto.


In [17]:
import matplotlib.pyplot as plt

FIGURES_DIR = BASE_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

route_figure_path = FIGURES_DIR / "rotas-otimizadas-final-clean.png"
history_figure_path = FIGURES_DIR / "convergencia-ga-final.png"

# figura de rotas mais limpa
stop_lookup = {stop.stop_id: stop for stop in stops}
depot = next(stop for stop in stops if stop.is_depot)

plt.figure(figsize=(10, 7))

for route in final_result.routes:
    x_points = [depot.x]
    y_points = [depot.y]

    for stop_id in route.stop_ids:
        stop = stop_lookup[stop_id]
        x_points.append(stop.x)
        y_points.append(stop.y)

    x_points.append(depot.x)
    y_points.append(depot.y)

    plt.plot(x_points, y_points, marker="o", linewidth=2, label=route.vehicle_id)

plt.scatter(depot.x, depot.y, s=180, c="black", marker="*", label="depot")
plt.text(depot.x + 0.15, depot.y + 0.15, "Depot", fontsize=10, fontweight="bold")

plt.title("Rotas otimizadas por veículo")
plt.xlabel("Coordenada X")
plt.ylabel("Coordenada Y")
plt.legend()
plt.tight_layout()
plt.savefig(route_figure_path, dpi=150)
plt.show()

# figura de convergencia
plot_history(final_result, history_figure_path)

print("Figura de rotas limpa salva em:", route_figure_path)
print("Figura de convergencia salva em:", history_figure_path)

Figura de rotas limpa salva em: /Volumes/SSD-EXT-MAC-FB/AI-Postgraduate/Tech Challenge B/Entregavel/fase-02/tech-challenge/figures/rotas-otimizadas-final-clean.png
Figura de convergencia salva em: /Volumes/SSD-EXT-MAC-FB/AI-Postgraduate/Tech Challenge B/Entregavel/fase-02/tech-challenge/figures/convergencia-ga-final.png


## 11. Leitura final da solução

A configuração final produziu uma distribuição equilibrada de entregas entre os veículos, eliminando os excessos de carga e mantendo apenas um excesso residual mínimo de distância em um dos veículos. Esse comportamento foi considerado aceitável para fins acadêmicos, dado o contexto de simulação e a necessidade de preservar a comparabilidade entre os experimentos.

Os gráficos complementares deste notebook ajudam a interpretar melhor o comportamento da solução final, mostrando a distribuição da amostra, a comparação entre experimentos e a carga e distância consolidadas por veículo.
